# RR_best dual-panel figure (v4)

更新内容：
- 直接读取 NSGA-II `summary.csv`（当前默认使用 `bal6` 全量结果），不再依赖旧的 `gwr_nsga2_optimized_results_with_risk_level_and_province.csv`。
- 在本 notebook 内部根据 `y_pred_base / y_pred_opt_best` 现场重建 `risk_level_base / risk_level_opt_best / risk_level_change`。
- 图 a：31 省统一计算 RR_best 平均值与 1/2/3/4 比例缩放，不对任何省份做特殊处理。
- 图 b：按总降低面积排序，省名与柱体平行，标签距柱顶约 3 mm。
- 输出文件名自动带 run tag，避免覆盖旧图。


In [ ]:

import os
import glob
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator, PercentFormatter

# =========================================================
# 模块 1：参数与路径（按需手动修改）
# =========================================================
ssp = "ssp2"        # TODO: 'hist' / 'ssp1' / 'ssp2' / 'ssp3' / 'ssp4' / 'ssp5'
ssp_time = "2080"   # TODO: '2020' / '2040' / '2060' / '2080' / '2100'
resolution = "10km" # TODO: '0.1km' / '1km' / '3km' / '5km' / '10km' / '20km'
RUN_TAG = "fast_n24_g16_inc_tot_s10_r42_bal11"

home_dir = os.path.expanduser("/path/to/home/")
base_path = os.path.join(home_dir, "PROJECT_DIR", "Papers", "4_NC")
path_name = f"Points_China_all_{resolution}"
print("path_name:", path_name)

optimization_dir = os.path.join(base_path, "outputs", path_name, ssp, ssp_time, "optimization")
out_dir = os.path.join(optimization_dir, "figure")
print("optimization_dir:", optimization_dir)
print("out_dir:", out_dir)

summary_candidates = []
if RUN_TAG:
    summary_candidates.append(os.path.join(optimization_dir, f"nsga2_{RUN_TAG}_summary.csv"))
summary_candidates.extend(
    p for p in sorted(glob.glob(os.path.join(optimization_dir, "nsga2_*_summary.csv")), key=os.path.getmtime, reverse=True)
    if p not in summary_candidates
)

SUMMARY_CSV_PATH = None
for candidate in summary_candidates:
    if os.path.exists(candidate):
        SUMMARY_CSV_PATH = candidate
        break
if SUMMARY_CSV_PATH is None:
    searched = "\n".join(summary_candidates) if summary_candidates else optimization_dir
    raise FileNotFoundError(f"No NSGA-II summary CSV found. Checked:\n{searched}")

print("SUMMARY_CSV_PATH:", SUMMARY_CSV_PATH)

OUTDIR = Path(out_dir)
OUTDIR.mkdir(parents=True, exist_ok=True)

FIG_TAG = RUN_TAG if RUN_TAG else Path(SUMMARY_CSV_PATH).stem.replace("nsga2_", "")
SUMMARY_OUT_PATH = OUTDIR / "rrbest_summary.csv"
PANEL_A_BASE = OUTDIR / "rrbest_c"
PANEL_B_BASE = OUTDIR / "rrbest_b"
COMBINED_BASE = OUTDIR / "rrbest_ab"

# =========================================================
# 模块 2：绘图风格
# =========================================================
palette = {
    0: "#E8E3DC",  # no decrease / unchanged
    1: "#8FA7B8",  # dusty blue
    2: "#A7BFA5",  # sage
    3: "#C8B39B",  # warm beige
    4: "#C99896",  # muted rose
}
background_bar = "#EEE8E1"
outline = "#6F6A68"
grid = "#D9D4CF"
white = "#FFFFFF"
text_gray = "#3D3A38"

PANEL_A_SIZE = (18.0 / 2.54, 6.0 / 2.54)
PANEL_B_SIZE = (8.0 / 2.54, 10.2 / 2.54)

def set_style():
    mpl.rcParams.update({
        "font.family": "Times New Roman",
        "font.size": 9,
        "axes.labelsize": 9,
        "axes.titlesize": 9,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
        "axes.linewidth": 0.6,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "savefig.facecolor": "white",
        "figure.facecolor": "white",
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.edgecolor": outline,
        "text.color": text_gray,
        "axes.labelcolor": text_gray,
        "xtick.color": text_gray,
        "ytick.color": text_gray,
    })

def save_fig(fig, basepath):
    svg_path = Path(f"{basepath}.svg")
    png_path = Path(f"{basepath}.png")
    fig.savefig(svg_path, format="svg")
    if png_path.exists():
        png_path.unlink()
    plt.close(fig)
    return svg_path

def cleanup_old_figure_exports(outdir: Path):
    patterns = [
        "risk_change_rrbest_panel_a_*.svg",
        "risk_change_rrbest_panel_a_*.png",
        "risk_change_rrbest_panel_b_*.svg",
        "risk_change_rrbest_panel_b_*.png",
        "risk_change_rrbest_dual_panel_combined_*.svg",
        "risk_change_rrbest_dual_panel_combined_*.png",
        "rrbest_a.svg",
        "rrbest_a.png",
        "rrbest_c.png",
        "rrbest_b.png",
        "rrbest_ab.png",
    ]
    for pattern in patterns:
        for path in Path(outdir).glob(pattern):
            try:
                path.unlink()
            except FileNotFoundError:
                pass

# =========================================================
# 模块 3：数据读取与风险等级重建
# =========================================================
import sys
if base_path not in sys.path:
    sys.path.insert(0, base_path)

from code.mgtwr.gwr_sigmoid_utils import gwr_scores_to_probabilities

def load_probability_boundary_config(base_path: str):
    train_gwr_dir = Path(base_path) / "code" / "3_gwr_model_train" / "national" / "GWR"
    boundary_candidates = [
        train_gwr_dir / "class_boundaries.pkl",
        Path(base_path) / "code" / "3_gwr_model_train" / "national" / "class_boundaries.pkl",
    ]
    artifact_candidates = [
        train_gwr_dir / "gwr_classification_artifacts.pkl",
    ]

    class_breaks = None
    transform_metadata = None
    boundary_source = None
    transform_source = None

    for candidate in boundary_candidates:
        if not candidate.exists():
            continue
        try:
            with candidate.open("rb") as f:
                boundary_data = pickle.load(f)
            candidate_breaks = np.asarray(boundary_data.get("class_breaks", []), dtype=float).reshape(-1)
            if candidate_breaks.size >= 2 and np.isfinite(candidate_breaks).all():
                class_breaks = np.sort(candidate_breaks)
                boundary_source = candidate
                candidate_transform = boundary_data.get("transform_metadata")
                if candidate_transform is not None:
                    transform_metadata = candidate_transform
                    transform_source = candidate
                break
        except Exception as e:
            print(f"[Warning] Failed to load class boundaries from {candidate}: {e}")

    if class_breaks is None:
        searched = "\n".join(str(x) for x in boundary_candidates)
        raise FileNotFoundError(
            "Failed to load class_boundaries.pkl from the GWR training outputs. Checked:\n"
            f"{searched}"
        )

    if transform_metadata is None:
        for candidate in artifact_candidates:
            if not candidate.exists():
                continue
            try:
                with candidate.open("rb") as f:
                    artifact_data = pickle.load(f)
                candidate_transform = artifact_data.get("transform_metadata")
                if candidate_transform is not None:
                    transform_metadata = candidate_transform
                    transform_source = candidate
                    break
            except Exception as e:
                print(f"[Warning] Failed to load transform metadata from {candidate}: {e}")

    return class_breaks, transform_metadata, boundary_source, transform_source


def ensure_probability_series(series: pd.Series, transform_metadata, series_name: str) -> pd.Series:
    values = pd.to_numeric(series, errors="coerce")
    probabilities = pd.Series(np.nan, index=series.index, dtype=float)
    valid = values.notna()
    if not valid.any():
        return probabilities

    finite_values = values[valid].to_numpy(dtype=float)
    if np.nanmin(finite_values) >= 0.0 and np.nanmax(finite_values) <= 1.0:
        probabilities.loc[valid] = np.clip(finite_values, 0.0, 1.0)
        print(f"{series_name}: detected probability input, clip to [0, 1].")
        return probabilities

    if transform_metadata is None:
        raise ValueError(
            f"{series_name}: detected raw GWR scores but no transform_metadata is available for sigmoid mapping."
        )

    mapped = gwr_scores_to_probabilities(
        finite_values,
        transform_metadata=transform_metadata,
        return_metadata=False,
    )
    probabilities.loc[valid] = np.clip(np.asarray(mapped, dtype=float), 0.0, 1.0)
    print(f"{series_name}: detected raw GWR scores, applied robust sigmoid mapping.")
    return probabilities


def classify_with_breaks(probabilities: pd.Series, class_breaks: np.ndarray) -> pd.Series:
    probs = pd.to_numeric(probabilities, errors="coerce")
    probs = probs.clip(lower=float(class_breaks[0]), upper=float(class_breaks[-1]))
    labels = pd.Series(pd.array([pd.NA] * len(probs), dtype="Int64"), index=probabilities.index)
    valid = probs.notna()
    labels.loc[valid] = (
        np.digitize(probs[valid].to_numpy(dtype=float), class_breaks[1:-1], right=True) + 1
    ).astype(int)
    return labels


df = pd.read_csv(SUMMARY_CSV_PATH)
required_cols = {"NAME_EN_JX", "RR_best", "y_pred_base", "y_pred_opt_best"}
missing = required_cols.difference(df.columns)
if missing:
    raise ValueError(f"Missing required columns in summary CSV: {missing}")

class_breaks, transform_metadata, boundary_source, transform_source = load_probability_boundary_config(base_path)
print("class_breaks:", class_breaks)
print("boundary source:", boundary_source)
print("transform source:", transform_source if transform_source is not None else "not needed / unavailable")

province_df = df[df["NAME_EN_JX"].notna()].copy()
province_df["y_prob_base"] = ensure_probability_series(province_df["y_pred_base"], transform_metadata, "y_pred_base")
province_df["y_prob_opt_best"] = ensure_probability_series(province_df["y_pred_opt_best"], transform_metadata, "y_pred_opt_best")
province_df["risk_level_base"] = classify_with_breaks(province_df["y_prob_base"], class_breaks)
province_df["risk_level_opt_best"] = classify_with_breaks(province_df["y_prob_opt_best"], class_breaks)
province_df["risk_level_change"] = (province_df["risk_level_base"] - province_df["risk_level_opt_best"]).astype("Int64")

print("risk_level_change value counts:")
print(province_df["risk_level_change"].value_counts(dropna=False).sort_index())

agg = province_df.groupby("NAME_EN_JX", dropna=True).agg(
    rr_mean=("RR_best", "mean"),
    n_cells=("RR_best", "size"),
)

agg["count_0"] = province_df.loc[province_df["risk_level_change"] <= 0].groupby("NAME_EN_JX").size()

for lvl in [1, 2, 3, 4]:
    agg[f"count_{lvl}"] = province_df.loc[province_df["risk_level_change"] == lvl].groupby("NAME_EN_JX").size()

agg = agg.fillna(0)
agg["positive_total"] = agg[[f"count_{i}" for i in [1, 2, 3, 4]]].sum(axis=1)
agg["positive_rate_pct"] = np.where(agg["n_cells"] > 0, agg["positive_total"] / agg["n_cells"] * 100.0, 0.0)

for lvl in [0, 1, 2, 3, 4]:
    agg[f"rate_{lvl}"] = np.where(agg["n_cells"] > 0, agg[f"count_{lvl}"] / agg["n_cells"] * 100.0, 0.0)

for lvl in [1, 2, 3, 4]:
    agg[f"share_{lvl}"] = np.where(
        agg["positive_total"] > 0,
        agg[f"count_{lvl}"] / agg["positive_total"],
        0.0
    )
    agg[f"scaled_rr_{lvl}"] = agg["rr_mean"] * agg[f"share_{lvl}"]

agg = agg.reset_index()

agg_rr = agg.sort_values(
    ["positive_rate_pct", "rr_mean", "positive_total", "NAME_EN_JX"],
    ascending=[False, False, False, True]
).reset_index(drop=True)
agg_rr["rr_order"] = np.arange(1, len(agg_rr) + 1)

agg_area = agg.sort_values(
    ["positive_total", "rr_mean", "NAME_EN_JX"],
    ascending=[False, False, True]
).reset_index(drop=True)
agg_area["area_order"] = np.arange(1, len(agg_area) + 1)

summary = agg.merge(agg_rr[["NAME_EN_JX", "rr_order"]], on="NAME_EN_JX", how="left")
summary = summary.merge(agg_area[["NAME_EN_JX", "area_order"]], on="NAME_EN_JX", how="left")
summary = summary.sort_values("rr_order").reset_index(drop=True)
summary.to_csv(SUMMARY_OUT_PATH, index=False)

# =========================================================
# 模块 4：图 b 标签辅助函数
# =========================================================
def add_radial_labels_figure(fig, ax, theta_vals, r_tops, names, offset_mm=3.0, fs_default=4.25):
    """
    省名与对应柱子平行，文字距柱顶约 offset_mm（默认 3 mm）。
    """
    fig.canvas.draw()
    for theta, top, name in zip(theta_vals, r_tops, names):
        p_center = ax.transData.transform((theta, 0))
        p_top = ax.transData.transform((theta, top))
        vec = p_top - p_center
        norm = np.hypot(vec[0], vec[1])
        if norm == 0:
            continue
        unit = vec / norm

        offset_px = offset_mm / 25.4 * fig.dpi
        p_text = p_top + unit * offset_px
        x_fig, y_fig = fig.transFigure.inverted().transform(p_text)

        angle = np.degrees(np.arctan2(unit[1], unit[0]))
        rotation = angle
        ha = "left"
        if angle > 90 or angle < -90:
            rotation = angle + 180
            ha = "right"

        fs = fs_default
        if len(name) >= 12:
            fs = fs_default * (4.4 / 5.4)
        elif len(name) >= 10:
            fs = fs_default * (4.9 / 5.4)

        fig.text(
            x_fig, y_fig, name,
            ha=ha, va="center",
            rotation=rotation, rotation_mode="anchor",
            fontsize=fs, color=text_gray
        )

def add_area_tick_labels_near_bar(fig, ax, theta, r_values, labels, offset_mm=3.0, fontsize=4.8):
    """把面积刻度标签放在指定柱子旁边，点位沿直径分布，但文字保持水平显示。"""
    fig.canvas.draw()
    p0 = ax.transData.transform((theta, 0.0))
    p1 = ax.transData.transform((theta, 1.0))
    radial = p1 - p0
    norm = np.hypot(radial[0], radial[1])
    if norm == 0:
        return
    radial = radial / norm
    tangent = np.array([-radial[1], radial[0]])
    if tangent[0] < 0:
        tangent = -tangent
    offset_px = offset_mm / 25.4 * fig.dpi
    for r, label in zip(r_values, labels):
        p_text = ax.transData.transform((theta, r)) + tangent * offset_px
        x_fig, y_fig = fig.transFigure.inverted().transform(p_text)
        fig.text(
            x_fig, y_fig, label,
            ha="center", va="center",
            fontsize=fontsize, fontweight="bold", color=text_gray
        )

# =========================================================
# 模块 5：图 a
# =========================================================
def plot_panel_a(data, basepath):
    set_style()
    fig, ax = plt.subplots(figsize=PANEL_A_SIZE)

    x = np.arange(len(data))
    width = 0.72
    bottoms = np.zeros(len(data), dtype=float)

    # 只堆叠风险等级降低的比例；level 0 作为图例说明 no decrease / unchanged。
    for lvl in [1, 2, 3, 4]:
        vals = data[f"rate_{lvl}"].to_numpy(dtype=float)
        ax.bar(
            x, vals, width=width, bottom=bottoms,
            color=palette[lvl], edgecolor="white", linewidth=0.45,
            zorder=3
        )
        bottoms += vals

    max_positive = float(np.nanmax(data["positive_rate_pct"])) if len(data) else 0.0
    ymax = 100.0
    ax.set_xlim(-0.6, len(data) - 0.4)
    ax.set_ylim(0, ymax)
    ax.set_ylabel("Risk level change rate")
    ax.yaxis.set_major_locator(MultipleLocator(20))
    ax.yaxis.set_major_formatter(PercentFormatter(xmax=100, decimals=0))

    ax.set_xticks(x)
    ax.set_xticklabels(data["NAME_EN_JX"], rotation=45, ha="right", va="top", rotation_mode="anchor")
    ax.tick_params(axis="x", labelsize=9, pad=0)
    ax.tick_params(axis="y", labelsize=9)

    ax.grid(axis="y", color=grid, linewidth=0.5, zorder=0)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    handles = [Patch(facecolor=palette[lvl], edgecolor="none", label=f"Δ level = {lvl}") for lvl in [1, 2, 3, 4]]
    leg = ax.legend(
        handles=handles,
        frameon=False, loc="upper right", ncol=5, bbox_to_anchor=(1.0, 1.01),
        handlelength=1.1, columnspacing=0.9, handletextpad=0.35, borderaxespad=0.0,
        fontsize=9
    )

    print(f"Panel a y-axis upper bound: {ymax:.0f}% (max province rate: {max_positive:.2f}%)")

    fig.subplots_adjust(left=0.075, right=0.995, bottom=0.35, top=0.82)
    return save_fig(fig, basepath)

# =========================================================
# 模块 6：图 b
# =========================================================
def plot_panel_b(data, basepath):
    # 设置全局绘图风格（字体、线宽、颜色等）
    set_style()

    # 创建约 8 cm 宽的独立环形图
    fig = plt.figure(figsize=PANEL_B_SIZE)

    # 添加极坐标轴（polar），并调整位置
    ax = fig.add_axes([0.05, 0.36, 0.82, 0.56], projection="polar")

    # 数据条数量与面积换算：10km 网格对应 100 km²。
    n = len(data)
    cell_area_km2 = float(str(resolution).lower().replace("km", "")) ** 2

    # 设置极坐标留白角度（形成开口）
    gap_deg = 30
    total_deg = 360 - gap_deg

    # 每个柱子的角度间隔（弧度制）
    step = np.deg2rad(total_deg / n)

    # 柱宽（略小于间隔，避免重叠）
    bar_width = step * 0.82

    # 起始角度（控制整体旋转位置）
    theta_start = np.deg2rad(190)

    # 每个柱子的角度位置
    thetas = theta_start + np.arange(n) * step

    # 设置角度方向为顺时针
    ax.set_theta_direction(-1)

    # 设置0°方向在正上方
    ax.set_theta_offset(np.pi / 2.0)

    # -----------------------------
    # 半径结构设置（视觉关键）
    # -----------------------------
    inner_radius = 0.92              # 内圈半径（中间空心大小）
    max_outer_thickness = 3.55       # 外圈最大厚度

    # 数据最大值（用于缩放）
    max_total = (data["positive_total"] * cell_area_km2).max()

    # 缩放因子（将数据映射到可视范围）
    scale = max_outer_thickness / max_total if max_total > 0 else 1.0

    # 初始柱子底部（从内圈开始）
    bottoms = np.full(n, inner_radius)

    # -----------------------------
    # 绘制堆叠柱（4个等级）
    # -----------------------------
    for lvl in [1, 2, 3, 4]:
        # 当前等级高度（按比例缩放）
        heights = data[f"count_{lvl}"].to_numpy() * cell_area_km2 * scale

        # 绘制极坐标柱状图（堆叠）
        ax.bar(
            thetas, heights,
            width=bar_width,
            bottom=bottoms,            # 堆叠关键：从上一个层开始
            color=palette[lvl],
            edgecolor="white",
            linewidth=0.0044,
            align="edge",
            zorder=1
        )

        # 更新底部位置（为下一层做准备）
        bottoms += heights

    # -----------------------------
    # 绘制外轮廓（总量边界）
    # -----------------------------
    total_heights = data["positive_total"].to_numpy() * cell_area_km2 * scale
    outer_tops = inner_radius + total_heights

    ax.bar(
        thetas,
        total_heights,
        width=bar_width,
        bottom=np.full(n, inner_radius),
        facecolor="none",             # 透明填充，只画边框
        edgecolor=outline,
        linewidth=0.50,
        align="edge",
        zorder=1.5
    )

    # -----------------------------
    # 设置半径范围（为标签预留空间）
    # -----------------------------
    provisional_outer = outer_tops.max() + 1.08
    ax.set_ylim(0, provisional_outer)

    # -----------------------------
    # 添加径向参考线（刻度）
    # -----------------------------
    # 非均匀刻度：前密后疏，显示 2 / 5 / 9 / 14 / 20 (×10^3 km²)
    guide_values = np.array([2000, 5000, 9000, 14000, 20000])

    # 过滤掉超过最大值的刻度
    guide_values = guide_values[guide_values <= max_total + 60]

    # 转换为实际半径位置
    guide_r = inner_radius + guide_values * scale

    # 设置刻度位置
    ax.set_rticks(guide_r.tolist())

    # 隐藏默认刻度标签；稍后手动放到 Inner Mongolia 柱子旁边。
    ax.set_yticklabels([""] * len(guide_r))

    # 绘制径向网格线
    ax.yaxis.grid(True, color=white, linewidth=0.38, zorder=2)

    # 关闭角度方向网格
    ax.xaxis.grid(False)

    # 移除角度刻度
    ax.set_xticks([])

    # 隐藏极坐标外框
    ax.spines["polar"].set_visible(False)

    names = data["NAME_EN_JX"].astype(str).tolist()
    if "Inner Mongolia" in names and len(guide_r) > 0:
        im_idx = names.index("Inner Mongolia")
        add_area_tick_labels_near_bar(
            fig,
            ax,
            thetas[im_idx] + bar_width / 2,
            guide_r,
            [f"{int(v / 1000)}" for v in guide_values],
            offset_mm=3.0,
            fontsize=4.8
        )

    # -----------------------------
    # 添加外侧标签（国家/地区名称）
    # -----------------------------
    add_radial_labels_figure(
        fig,
        ax,
        thetas + bar_width / 2,   # 标签位于柱子中心
        outer_tops,               # 标签基于柱顶位置
        data["NAME_EN_JX"].tolist(),
        offset_mm=0.45,
        fs_default=3.24           # 原 5.4 的 0.6 倍
    )

    # -----------------------------
    # 添加图面文字
    # -----------------------------
    # fig.text(
    #     0.03, 0.96, "b",
    #     fontsize=9,
    #     fontweight="bold",
    #     ha="left",
    #     va="top",
    #     color=text_gray
    # )

    fig.text(
        0.43, 0.21,
        "×10³ km²",
        ha="center",
        va="center",
        fontsize=4.8,
        color=text_gray
    )

    fig.text(
        0.55, 0.11,
        "",
        ha="center",
        va="center",
        fontsize=9,
        color=text_gray
    )

    # -----------------------------
    # 图例（4个等级）
    # -----------------------------
    handles = [
        Patch(facecolor=palette[lvl], edgecolor="none", label=f"Δ level = {lvl}")
        for lvl in [1, 2, 3, 4]
    ]

    fig.legend(
        handles=handles,
        title="Area with reduced risk",
        loc="center left",
        bbox_to_anchor=(0.64, 0.22),
        ncol=1,
        frameon=False,
        handlelength=1.1,
        handletextpad=0.35,
        columnspacing=0.5,
        borderaxespad=0.0,
        prop={"size": 4.8},
        title_fontsize=5.1
    )

    # 保存图像并返回路径
    return save_fig(fig, basepath)
# =========================================================
# 模块 7：组合图
# =========================================================
def plot_combined(data_a, data_b, basepath):
    set_style()
    fig = plt.figure(figsize=FIGSIZE_COMBINED)

    # panel a
    ax1 = fig.add_axes([0.05, 0.20, 0.42, 0.68])
    x = np.arange(len(data_a))
    width = 0.72

    ax1.bar(
        x, data_a["rr_mean"], width=width,
        color=background_bar, edgecolor=outline, linewidth=0.6,
        zorder=2
    )

    bottoms = np.zeros(len(data_a))
    for lvl in [1, 2, 3, 4]:
        vals = data_a[f"scaled_rr_{lvl}"].to_numpy()
        ax1.bar(
            x, vals, width=width, bottom=bottoms,
            color=palette[lvl], edgecolor="white", linewidth=0.45, zorder=3
        )
        bottoms += vals

    ymax = max(0.6, data_a["rr_mean"].max() * 1.12)
    ax1.set_xlim(-0.6, len(data_a) - 0.4)
    ax1.set_ylim(0, ymax)
    ax1.set_ylabel("Mean RR_best")
    ax1.set_xticks(x)
    ax1.set_xticklabels(data_a["NAME_EN_JX"], rotation=58, ha="right", fontsize=5.8)
    ax1.yaxis.set_major_locator(MultipleLocator(0.1))
    ax1.grid(axis="y", color=grid, linewidth=0.5, zorder=0)
    ax1.spines["top"].set_visible(False)
    ax1.spines["right"].set_visible(False)
    ax1.text(-0.08, 1.02, "a", transform=ax1.transAxes, fontsize=9, fontweight="bold", ha="left", va="bottom")

    handles = [Patch(facecolor=palette[lvl], edgecolor="none", label=f"{lvl}") for lvl in [1, 2, 3, 4]]
    leg = ax1.legend(
        handles=handles, title="Decrease level", title_fontsize=5.8,
        frameon=False, loc="upper right", ncol=2, bbox_to_anchor=(1.0, 1.02),
        handlelength=0.85, columnspacing=0.8, handletextpad=0.35, borderaxespad=0.1,
        fontsize=5.6
    )
    plt.setp(leg.get_title(), color=text_gray)

    # panel b
    ax2 = fig.add_axes([0.54, -0.02, 0.41, 0.93], projection="polar")

    n = len(data_b)
    gap_deg = 30
    total_deg = 360 - gap_deg
    step = np.deg2rad(total_deg / n)
    bar_width = step * 0.82
    theta_start = np.deg2rad(190)
    thetas = theta_start + np.arange(n) * step

    ax2.set_theta_direction(-1)
    ax2.set_theta_offset(np.pi / 2.0)

    inner_radius = 0.92
    max_outer_thickness = 3.55
    max_total = data_b["positive_total"].max()
    scale = max_outer_thickness / max_total if max_total > 0 else 1.0

    bottoms = np.full(n, inner_radius)
    for lvl in [1, 2, 3, 4]:
        heights = data_b[f"count_{lvl}"].to_numpy() * scale
        ax2.bar(
            thetas, heights, width=bar_width, bottom=bottoms,
            color=palette[lvl], edgecolor="white", linewidth=0.44,
            align="edge", zorder=3
        )
        bottoms += heights

    total_heights = data_b["positive_total"].to_numpy() * scale
    outer_tops = inner_radius + total_heights
    ax2.bar(
        thetas, total_heights, width=bar_width, bottom=np.full(n, inner_radius),
        facecolor="none", edgecolor=outline, linewidth=0.50,
        align="edge", zorder=4
    )

    provisional_outer = outer_tops.max() + 1.08
    ax2.set_ylim(0, provisional_outer)

    guide_values = np.array([200,400, 600,800,1000, 1200])
    guide_values = guide_values[guide_values <= max_total + 60]
    guide_r = inner_radius + guide_values * scale
    ax2.set_rticks(guide_r.tolist())
    ax2.set_yticklabels([""] * len(guide_r))
    ax2.yaxis.grid(True, color=grid, linewidth=0.38)
    ax2.xaxis.grid(False)
    ax2.set_xticks([])
    ax2.spines["polar"].set_visible(False)

    add_radial_labels_figure(
        fig, ax2,
        thetas + bar_width / 2,
        outer_tops,
        data_b["NAME_EN_JX"].tolist(),
        offset_mm=3.0,
        fs_default=3.95
    )

    fig.text(0.51, 0.96, "(b)", fontsize=9, fontweight="bold", ha="left", va="top", color=text_gray)
    fig.text(0.75, 0.46, "Total area of decrease", ha="center", va="center", fontsize=5.2, color=text_gray)
    fig.text(0.75, 0.38, "(×100 km²)", ha="center", va="center", fontsize=5.0, color=text_gray)

    return save_fig(fig, basepath)

# =========================================================
# 模块 8：运行与导出
# =========================================================
cleanup_old_figure_exports(OUTDIR)
plot_panel_a(agg_rr, PANEL_A_BASE)
plot_panel_b(agg_area, PANEL_B_BASE)
# 按当前制图要求，只输出 panel a 和 panel b，不再生成组合图。

print("Data file:", SUMMARY_CSV_PATH)
print("Generated:")
print(SUMMARY_OUT_PATH)
print(Path(f"{PANEL_A_BASE}.svg"))
print(Path(f"{PANEL_B_BASE}.svg"))
print("Combined panel skipped by current plotting settings.")
